| Étape | Description                                             |
| ----- | ------------------------------------------------------- |
| 1️   | Avoir les requêtes, documents, et pertinences (`qrels`) |
| 2️   | Générer les embeddings avec un modèle                   |
| 3️   | Calculer la similarité (ex. : cosine)                   |
| 4️   | Classer les documents et créer `run_dict`               |
| 5️   | Évaluer avec `Ranx` en choisissant les métriques        |


In [ ]:
qrels_dict = {
    "q1": {"d1": 1, "d3": 1},
    "q2": {"d2": 1},
    "q3": {"d4": 1, "d5": 1},
}


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

queries = ["Quelle est la capitale de la France ?", "Qui est le président ?", "Définition de l’IA"]
docs = ["Paris est la capitale de la France.",
        "Le président actuel est Emmanuel Macron.",
        "L’intelligence artificielle est un domaine technologique."]

query_ids = ["q1", "q2", "q3"]
doc_ids = ["d1", "d2", "d3"]

# Embeddings
query_embeddings = model.encode(queries, convert_to_numpy=True)
doc_embeddings = model.encode(docs, convert_to_numpy=True)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Similarité
sims = cosine_similarity(query_embeddings, doc_embeddings)

# Génération du run_dict
run_dict = {}
for i, q_id in enumerate(query_ids):
    ranked_scores = {
        doc_ids[j]: float(sims[i][j]) for j in np.argsort(sims[i])[::-1]
    }
    run_dict[q_id] = ranked_scores


In [ ]:
from ranx import Qrels, Run, evaluate

# Création des objets Ranx
qrels = Qrels(qrels_dict)
run = Run(run_dict, name="mon_run")

# Liste des métriques
metrics = ["mrr@10", "ndcg@10", "map@10", "precision@1", "recall@10"]

# Évaluation
results = evaluate(qrels, run, metrics=metrics)
print(results)
